In [38]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict , Annotated
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
from pydantic import BaseModel , Field
import operator

In [39]:
load_dotenv()


True

In [55]:
model = ChatOpenAI(
    model="google/gemini-2.0-flash-exp:free",
    api_key =os.getenv("OPEN_AI_API_KEY"),
    base_url =os.getenv("OPENAI_BASE_URL")
     
)

In [41]:
class evaluation_schema(BaseModel):
    feedback: str = Field(description="Detailed feedback of the essay")
    score: int = Field(description = "Score out of 10" , ge=0 , le= 10)
    

In [42]:
structured_model = model.with_structured_output(evaluation_schema)


In [16]:
essay = f"""
**Democracy**

Democracy is a system of government in which the people have the power to choose their leaders.
It allows citizens to vote and participate in decision-making.
Democracy promotes freedom, equality, and justice for all people.
In a democratic country, everyone has the right to express their opinions.
The government is accountable to the people and must follow the law.
Free and fair elections are an important feature of democracy.
It protects the rights and liberties of citizens.
Democracy encourages public participation in national affairs.
It helps maintain peace and stability in society.
Therefore, democracy is considered one of the best forms of government.

"""

In [21]:
prompt = f'Evaluate the language quality of the following essay and give the score out of 10. {essay}'
model.invoke(prompt)


AIMessage(content='User Safety: safe', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 172, 'prompt_tokens': 603, 'total_tokens': 775, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 198, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3.5-content-safety-20260604:free', 'system_fingerprint': None, 'id': 'gen-1780682073-Qt0kmE46nmJC4J6QYwND', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e98ec-2aa1-7780-b1d1-237470b6a527-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 603, 'output_tokens': 172, 'total_tokens': 775, 'inpu

In [43]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_score: Annotated[list[int] , operator.add]
    average_score: float
    
    

In [ ]:
def evaluate_language(state: UPSCState):
    prompt = f'Evaluate the language quality of the folowing essay and provide a  short feedback and assign a score out of 10 \n..{essay}'
    output = structured_model.invoke(prompt)
    
    return {'language_feedback' : output.feedback , 'individual_score' : [output.score]}


In [ ]:
def evaluate_analysis(state: UPSCState):
    prompt = f'Evaluate the depth of analysis of the folowing essay and provide a short feedback and assign a score out of 10 \n..{essay}'
    output = structured_model.invoke(prompt)
    
    return {'analysis_feedback' : output.feedback , 'individual_score' : [output.score]}


In [ ]:
def evaluate_thought(state: UPSCState):
    prompt = f'Evaluate the quality of thought of the folowing essay and provide a  short feedback and assign a score out of 10 \n..{essay}'
    output = structured_model.invoke(prompt)
    
    return {'clarity_feedback' : output.feedback , 'individual_score' : [output.score]}


In [ ]:
def final_evaluation(state: UPSCState):
    prompt = f'Based on the following feedback create a short feedback \n language_feedback - {state["language_feedback"] } \n analysis_feedback - {state["analysis_feedback"]} \n clarity_thought_feedback - {state["clarity_feedback"]}'
    overall_feedback = model.invoke(prompt).content
    
    
    #average
    avg_score =sum(state['individual_score']) / len(state['individual_score'])
    
    return {'overall_feedback' : overall_feedback , 'average_score' : avg_score}

     

In [53]:
graph = StateGraph(StateGraph)

graph.add_node('evaluate_language' , evaluate_language)
graph.add_node('evaluate_analysis' , evaluate_analysis)
graph.add_node('evaluate_thought' , evaluate_thought)
graph.add_node('final_evaluation' , final_evaluation)


#edges
graph.add_edge(START , 'evaluate_language')
graph.add_edge(START , 'evaluate_analysis')
graph.add_edge(START , 'evaluate_thought')

graph.add_edge('evaluate_language' , 'final_evaluation')
graph.add_edge('evaluate_analysis' , 'final_evaluation')
graph.add_edge('evaluate_thought' , 'final_evaluation')

graph.add_edge('final_evaluation' , END)


workflow = graph.compile()

In [ ]:
initial_state = {
    'essay' : essay
}
workflow.invoke(initial_state)